In [ ]:
# config
samples = ["A1", "A2", "B2", "C2", "D1"]
input_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/raw_data"
zarr_file_path = "/scratch/leuven/357/vsc35768/spatial-transcriptomics/intermediate_results/20260126_final.zarr"
plotting = True
on_hpc = True
unit_testing = False
gpu = False
outdir = "figures"

In [ ]:
import os, re
import numpy as np
import pandas as pd

import spatialdata as sd
import harpy as hp
import scanpy as sc
import scvi

from matplotlib.axes import Axes
import matplotlib.pyplot as plt
from matplotlib.collections import PathCollection
import matplotlib as mpl
import spatialdata_plot

mpl.rcParams["axes.linewidth"] = 1

## Reading in the data

In [ ]:
sdata = sd.read_zarr(
    zarr_file_path,
    on_bad_files = "warn"
)
sdata

In [ ]:
# loading the model
scvi_model = scvi.model.SCVI.load("intermediate_results/model_20260128_final")
# get the anndata from the model
adata_scvi = scvi_model.adata
adata_scvi

### QC: dotplot

In [ ]:
marker_genes_dict = {
    "Neurons": ["Rbfox3", "Tubb3", "Snap25", "Syt1"],
    "Excitatory Neurons": ["Slc17a7", "Slc17a6"],
    "Inhibitory Neurons": ["Gad1", "Gad2"],
    "Astrocytes": ["Slc1a3", "Slc1a2", "Aldh1l1", "Aldoc", "Aqp4"],
    "Microglia": ["Csf1r", "C1qa"],
    "Endothelial Cells": ["Flt1", "Pecam1"],
    "Oligodendrocytes": ["Plp1", "Mog"],
    "OPCs DEGs": ["Tnr", "Vcan"]
}

cell_types = [
    "CA1 Neurons",
    "CA2/CA3 Neurons",
    "DG Granular Neurons",
    "DG Subgranular Layer",
    "Thalamic Neurons",
    "Medial Habenula Neurons",
    "Cortical Neurons",
    "Inhibitory Neurons",
    "Astrocytes Protoplasmic",
    "Astrocytes Fibrous",
    "Microglia",
    "Endothelial Cells",
    "Choroid Plexus Cells",
    "Oligodendrocytes",
    "OPCs"
]

adata_scvi.obs["cell_type"] = adata_scvi.obs["cell_type"].astype("category")
adata_scvi.obs["cell_type"] = adata_scvi.obs["cell_type"].cat.reorder_categories(cell_types, ordered=True)

dp = sc.pl.dotplot(
    adata_scvi,
    var_names = marker_genes_dict,
    groupby = "cell_type",
    cmap = "BuGn",
    use_raw = False,
    return_fig = True,
    vmax = 2,
    show = False,
    figsize = (10, 5)
)
dp.style(dot_edge_color='black', dot_edge_lw=0).show()

dp.savefig("figures/dp_qc.svg", bbox_inches="tight")

### Helper: build one global color palette mapped to cell type

In [ ]:
# build one global color palette per cell type
import matplotlib.pyplot as plt
from matplotlib.colors import to_hex

def table_name_for(sample):
    return f"{sample}_transcriptomics_filter_scvi_annotated"

def get_15_darkish():
    cmaps = ["tab10", "Dark2", "Set1"]
    cols = []
    for name in cmaps:
        cols += [to_hex(c) for c in plt.get_cmap(name).colors]
    # unique preserve order
    seen, out = set(), []
    for c in cols:
        if c not in seen:
            out.append(c); seen.add(c)
    return out[:15]

palette15 = get_15_darkish()

# global ordered list of all cell types across samples
all_cts = []
for sample in samples:
    ad = sdata.tables[table_name_for(sample)]
    all_cts.extend(ad.obs["cell_type"].astype("category").cat.categories.tolist())

seen = set()
all_cts = [ct for ct in all_cts if not (ct in seen or seen.add(ct))]

ct2color = {ct: palette15[i % len(palette15)] for i, ct in enumerate(all_cts)}


### QC: spatial plots of cell types separately

In [ ]:
outdir = "figures"
os.makedirs(outdir, exist_ok=True)

def safe_name(x: str) -> str:
    return re.sub(r"[^A-Za-z0-9._-]+", "_", str(x))

sample = "A1"
label_key  = f"{sample}_segmentation_mask"
table_name = f"{sample}_transcriptomics_filter_scvi_annotated"
tmp_table = f"{sample}_tmp_plot_table"

adata0 = sdata.tables[table_name]
cell_types = adata0.obs[ct_col].astype("category").cat.categories

for ct in cell_types:
    adata = adata0.copy()
    focal = adata.obs[ct_col].astype(str)
    adata.obs["_ct_focal"] = focal.where(focal == ct, other="Other").astype("category")
    sdata.tables[tmp_table] = adata

    sdata.pl.render_labels(
        label_key,
        color = "_ct_focal",
        groups = [ct, "Other"],
        palette = [ct2color[str(ct)], "lightgray"],  # <- always from ct2color
        contour_px = None,
        fill_alpha = 1.0,
        outline_alpha = 0.0,
        scale = None,
        table_name = tmp_table,
        method = "matplotlib",
    ).pl.show(
        coordinate_systems = sample, 
        dpi = 600
    )

    fig = plt.gcf()
    fn = os.path.join(outdir, f"{sample}_{safe_name(ct)}.svg")
    fig.savefig(fn, format="svg", bbox_inches="tight", dpi=600)
    plt.close(fig)


### QC: Plot cell types in all samples

In [ ]:
for sample in samples:
    label_key  = f"{sample}_segmentation_mask"
    table_name = f"{sample}_transcriptomics_filter_scvi_annotated"

    ad = sdata.tables[table_name].copy()
    ad.obs[ct_col] = ad.obs[ct_col].astype("category")
    present = [ct for ct in all_cts if ct in set(ad.obs[ct_col].cat.categories)]
    palette = [ct2color[ct] for ct in present]

    sdata.tables[f"{sample}_tmp_allct"] = ad  # optional, can also use table_name directly

    sdata.pl.render_labels(
        label_key,
        color=ct_col,
        groups=present,
        palette=palette,
        na_color="lightgray",
        contour_px=None,
        fill_alpha=1.0,
        outline_alpha=0.0,
        scale=None,
        table_name=f"{sample}_tmp_allct",
        method="matplotlib",
    ).pl.show(coordinate_systems=sample, dpi=600)

    fig = plt.gcf()
    fn = os.path.join(outdir, f"{sample}.svg")
    fig.savefig(fn, format="svg", bbox_inches="tight", dpi=600)
    plt.close(fig)

### QC: UMAP of cell types

In [ ]:
import scanpy as sc

adata = adata_scvi.copy()
adata.obs[ct_col] = adata.obs[ct_col].astype("category")

present = [ct for ct in all_cts if ct in adata.obs[ct_col].cat.categories]
adata.obs[ct_col] = adata.obs[ct_col].cat.reorder_categories(present, ordered=True)

palette = [ct2color[ct] for ct in present]

umap_cell_types = sc.pl.umap(
    adata,
    color=ct_col,
    palette=palette,
    size=6,
    frameon=False,
    add_outline=False,
    save="_cell_types.svg",
    legend_fontsize = "xx-small",
    title = None
)


### QC: UMAP across samples

In [ ]:
# Setting the color for the samples in the next QCs
palette = dict(zip(samples, sns.color_palette("Set3", len(samples))))

In [ ]:
adata = adata_scvi.copy()

umap_cell_types = sc.pl.umap(
    adata,
    color="sample_id",
    palette = palette,
    size=6,
    frameon=False,
    add_outline=False,
    save="_samples.svg",
    legend_fontsize = "xx-small",
    title = None
)

### QC: n_genes, n_counts & n_cells per sample

In [ ]:
adata

In [ ]:
# x axis labels off
mpl.rcParams["xtick.labelbottom"] = False

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax = sc.pl.violin(
    adata,
    keys="n_counts",
    groupby="sample_id",
    palette = palette,
    stripplot=False,
    show=False,
    ax = ax
)

# compute means in the same group order as plotted
groups = adata.obs["sample_id"].astype("category").cat.categories
means = (
    adata.obs.groupby("sample_id")["n_counts"]
    .mean()
    .reindex(groups)
    .to_numpy()
)

# overlay mean as a horizontal marker per violin
ax.scatter(
    np.arange(len(groups)),
    means,
    marker="_",   # looks like a small line
    s=800,        # length/thickness
    zorder=10,
    color="black",
)

plt.tight_layout()
fn = os.path.join(outdir, "violin_n_counts_by_sample_id.svg")
plt.gcf().savefig(fn, format="svg", bbox_inches="tight")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax = sc.pl.violin(
    adata,
    keys="n_genes_by_counts",
    groupby="sample_id",
    palette = palette,
    stripplot=False,
    show=False,
    ax = ax
)

# compute means in the same group order as plotted
groups = adata.obs["sample_id"].astype("category").cat.categories
means = (
    adata.obs.groupby("sample_id")["n_genes_by_counts"]
    .mean()
    .reindex(groups)
    .to_numpy()
)

# overlay mean as a horizontal marker per violin
ax.scatter(
    np.arange(len(groups)),
    means,
    marker="_",   
    s=800,        
    zorder=10,
    color="black",
)

plt.tight_layout()
fn = os.path.join(outdir, "violin_n_genes_by_sample_id.svg")
plt.gcf().savefig(fn, format="svg", bbox_inches="tight")

plt.show()

In [ ]:
# counts per sample
counts = adata.obs["sample_id"].value_counts()

# keep categorical order if sample_id is categorical
if str(adata.obs["sample_id"].dtype) == "category":
    order = adata.obs["sample_id"].cat.categories
    counts = counts.reindex(order)

clr = plt.get_cmap("Set3").colors
bar_colors = [clr[i % len(clr)] for i in range(len(counts))]

fig, ax = plt.subplots(figsize=(7, 6))
ax.bar(
    counts.index.astype(str), 
    counts.values,
    color = bar_colors 
)
 
ax.set_ylabel("Number of cells")
ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
fn = os.path.join(outdir, "barplot_n_cells_by_sample_id.svg")
plt.gcf().savefig(fn, format="svg", bbox_inches="tight")

plt.show()

### QC: n_genes, n_counts & n_cells per cell type

In [ ]:
adata

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6)) 

ax = sc.pl.violin(
    adata,
    keys="n_counts",
    groupby="cell_type",
    stripplot=False,
    show=False,
    palette = ct2color,
    ax = ax
)

# compute means in the same group order as plotted
groups = adata.obs["cell_type"].astype("category").cat.categories
means = (
    adata.obs.groupby("cell_type")["n_counts"]
    .mean()
    .reindex(groups)
    .to_numpy()
)

# overlay mean as a horizontal marker per violin
ax.scatter(
    np.arange(len(groups)),
    means,
    marker="_",   # looks like a small line
    s=100,        # length/thickness
    zorder=10,
    color="black",
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

plt.tight_layout()
fn = os.path.join(outdir, "violin_n_counts_by_cell_type.svg")
plt.gcf().savefig(fn, format="svg", bbox_inches="tight")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6)) 

ax = sc.pl.violin(
    adata,
    keys="n_genes_by_counts",
    groupby="cell_type",
    stripplot=False,
    show=False,
    palette = ct2color,
    ax = ax
)

# compute means in the same group order as plotted
groups = adata.obs["cell_type"].astype("category").cat.categories
means = (
    adata.obs.groupby("cell_type")["n_genes_by_counts"]
    .mean()
    .reindex(groups)
    .to_numpy()
)

# overlay mean as a horizontal marker per violin
ax.scatter(
    np.arange(len(groups)),
    means,
    marker="_",   # looks like a small line
    s=100,        # length/thickness
    zorder=10,
    color="black",
)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

plt.tight_layout()
fn = os.path.join(outdir, "violin_n_genes_by_cell_type.svg")
plt.gcf().savefig(fn, format="svg", bbox_inches="tight")

plt.show()

In [ ]:
# counts per sample
counts = adata.obs["cell_type"].value_counts()

# keep categorical order if sample_id is categorical
if str(adata.obs["cell_type"].dtype) == "category":
    order = adata.obs["cell_type"].cat.categories
    counts = counts.reindex(order)

bar_colors = [ct2color.get(ct, "lightgray") for ct in counts.index]

fig, ax = plt.subplots(figsize=(7, 6))
ax.bar(
    counts.index.astype(str), 
    counts.values,
    color=bar_colors
    )

ax.set_ylabel("Number of cells")
ax.tick_params(axis="x", rotation=45)
ax.set_xticklabels(ax.get_xticklabels(), rotation=90)

plt.tight_layout()
fn = os.path.join(outdir, "barplot_n_cells_by_cell_type.svg")
plt.gcf().savefig(fn, format="svg", bbox_inches="tight")

plt.show()

### QC: distribution of cell type across samples

In [ ]:
ct_counts = pd.crosstab(adata.obs["sample_id"], adata.obs["cell_type"])
ct_pct = ct_counts.div(ct_counts.sum(axis=1), axis=0) * 100
ct_pct_long = (
    ct_pct.reset_index()
          .melt(id_vars="sample_id", var_name="cell_type", value_name="percent")
)
ct_pct_long

In [ ]:
colors = [ct2color[ct] for ct in ct_pct.columns]

fig, ax = plt.subplots(figsize=(7, 6))

ax = ct_pct.plot(
    kind="bar", 
    stacked=True, 
    figsize=(7, 6),
    color = colors,
    legend=False,
    width = 0.8,
    ax = ax
)
ax.set_xlabel(None)
plt.tight_layout()
fn = os.path.join(outdir, "stacked_plot_cell_type_per_sample.svg")
plt.gcf().savefig(fn, format="svg", bbox_inches="tight")

plt.show()

#ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")